In [ ]:
import os
import time
from dotenv import load_dotenv
import json
import re
import random
import string
import traceback
import logging
import asyncio
import sys
import pandas as pd 
sys.path.append(os.path.abspath('..'))
from utilities import PostgresClient

from daytona import Daytona, DaytonaConfig, CreateSandboxFromSnapshotParams
from openai import AzureOpenAI
load_dotenv()

# Initialize the Daytona client

# print(os.getenv("DAYTONA_KEY"))
daytona = Daytona(DaytonaConfig(api_key=os.getenv("DAYTONA_KEY")))
# Create the Sandbox instance
chat_client = AzureOpenAI(
    api_key=os.getenv("GPT5OMINIKEY"),
    api_version=os.getenv("OPENAI_API_VERSION_GPT_5O"),
    azure_endpoint=os.getenv("GPT5OMINIENDPOINT"),
)
pg_client = PostgresClient(
    host=os.getenv('DB_HOST'),
    username=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
    db_port=os.getenv('DB_PORT'),
    db_name=os.getenv('DB_NAME')
)
print("Daytona, PGClient, and AzureOpenAI clients initialized")


In [ ]:
sandbox = daytona.create()

# Run code securely inside the Sandbox
response = sandbox.process.code_run('print("Sum of 3 and 4 is " + str(3 + 4))')
if response.exit_code != 0:
    print(f"Error running code: {response.exit_code} {response.result}")
else:
    print(response.result)

# Clean up the Sandbox
sandbox.delete()

In [ ]:
def sample_large_text(text, max_chars=12000, num_samples=4):
    """Sample from the text to avoid hitting the max context window. Divided based on num_samples"""
    length = len(text)
    if length <= max_chars:
        return text
    
    sample_size = max_chars // num_samples
    samples = []
    
    # Distribute samples evenly across the text
    for i in range(num_samples):
        # Calculate the starting position as a percentage of total length
        # For 5 samples: 0%, 20%, 40%, 60%, 80%
        position_percent = i / num_samples
        start_pos = int(position_percent * length)
        
        # Make sure we don't go past the end
        end_pos = min(start_pos + sample_size, length)
        samples.append(text[start_pos:end_pos])
    
    return "\n\n... [content omitted] ...\n\n".join(samples)

In [ ]:
import json
from datetime import datetime

test_path = "/Users/kappavi/Documents/Avi's Documents /Work/bb-daytona/test/sample2.json"

# Step 1: Get user inputs
print("=" * 60)
raw_data = input("What do you want parsed? (Enter raw text/JSON or provide a relative path to a file): ")
# check if raw_data is a path, else parse it litrelaly
print(f"Raw data sample: (determining if path or raw data): {raw_data[:100]}")
if os.path.exists(raw_data):
    with open(raw_data, 'r') as file:
        raw_data = file.read()
        # raw_data = raw_data[:3000] # truncate super loong inputs 
        raw_data = sample_large_text(raw_data)
        print(f"Reading from inputted file: {raw_data[:100]}")
else:
    raw_data = sample_large_text(raw_data)
    print(f"Parsing raw data: {raw_data[:100]}")
expected_fields = input("What fields do you expect? (e.g., item_name, item_id, price, is_on_promotion): ")
print("=" * 60)

# Step 2: Use AI to generate parser code dynamically
prompt = f"""Generate Python code that will parse the following raw data and extract these field informations (They may not follow the exact same name or form,
 but the general idea should be the same): {expected_fields}
 For example, if the user was item_id, find a field that assigns some sort of id to an item.

Raw data:
{raw_data}

Requirements:
- The code should parse the data and create a JSON output with the requested fields
- Handle cases where fields might not be found (use None or appropriate defaults)
- The final line should print the result as a JSON string using json.dumps()
- Import any necessary libraries at the top
- Be smart about inferring the data structure and extracting the information

Only output the Python code, no explanations."""

time_start = time.time()
response = chat_client.chat.completions.create(
    model='gpt-5-mini',
    messages=[
        {"role": "system", "content": "You are a Python code generator. Output only valid Python code without any markdown formatting or explanations."},
        {"role": "user", "content": prompt}
    ]
)
time_end = time.time()
generated_code = response.choices[0].message.content.strip()
# Remove markdown code blocks if present
generated_code = re.sub(r'^```python\n|^```\n|```$', '', generated_code, flags=re.MULTILINE).strip()

print("\n📝 Generated Parser Code:")
print("-" * 60)
print(generated_code)
print("-" * 60)
print(f"Time taken to dynamically generate parser code: {time_end - time_start} seconds")

# Step 3: Run the parser in Daytona sandbox
print("\n🚀 Running parser in Daytona sandbox...")
params = CreateSandboxFromSnapshotParams(language="python")
sandbox = daytona.create(params)

try:
    time_start = time.time()
    result = sandbox.process.code_run(generated_code)
    
    if result.exit_code != 0:
        print(f"❌ Error running parser: {result.exit_code}")
        print(result.result)
    else:
        print("\n✅ Parser executed successfully!")
        print("\n📦 Parsed Output:")
        print("=" * 60)
        # Try to pretty print if it's valid JSON
        try:
            parsed_json = json.loads(result.result.strip())
            print(json.dumps(parsed_json, indent=2))
        except:
            print(result.result)
        print("=" * 60)
        time_end = time.time()
        print(f"Time taken to parse data: {time_end - time_start} seconds")
finally:
    # Step 4: Clean up
    sandbox.delete()
    print("\n🧹 Sandbox cleaned up")

# write teh outpout to a file 
today_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = f"output_{today_time}.json"
with open(output_file, 'w') as f:
    json.dump(parsed_json, f, indent=2)
print(f"Output written to {output_file}")

